# GitHub Actions & CI/CD for ML

## Overview
GitHub Actions is GitHub's built-in CI/CD platform. For ML projects it enables automated testing, model validation, and deployment pipelines. This notebook covers GitHub Actions from basics to ML-specific patterns.

---

## 1. GitHub Actions Concepts

```
Workflow (.github/workflows/*.yml)
  └── Job (runs on a runner machine)
        └── Step (individual command or action)
```

| Concept | Description |
|---------|-------------|
| **Workflow** | YAML file defining the automation |
| **Event/Trigger** | What starts the workflow |
| **Job** | Group of steps, runs on one runner |
| **Step** | Individual command or action |
| **Action** | Reusable step from the marketplace |
| **Runner** | VM that executes jobs |
| **Artifact** | File produced by a workflow |

---

## 2. Workflow Syntax

```yaml
name: CI Pipeline

on:                          # Triggers
  push:
    branches: [main, develop]
  pull_request:
    branches: [main]
  schedule:
    - cron: '0 2 * * 1'     # Every Monday at 2 AM
  workflow_dispatch:          # Manual trigger
    inputs:
      environment:
        type: choice
        options: [dev, staging, prod]

env:                         # Global environment variables
  PYTHON_VERSION: '3.11'

jobs:
  test:
    runs-on: ubuntu-latest   # Runner
    
    strategy:
      matrix:                # Run across multiple versions
        python-version: ['3.10', '3.11', '3.12']
    
    steps:
      - name: Checkout code
        uses: actions/checkout@v4
        with:
          fetch-depth: 0     # Full history (needed for DVC)
      
      - name: Setup Python
        uses: actions/setup-python@v5
        with:
          python-version: ${{ matrix.python-version }}
      
      - name: Cache dependencies
        uses: actions/cache@v4
        with:
          path: ~/.cache/pip
          key: ${{ runner.os }}-pip-${{ hashFiles('requirements.txt') }}
      
      - name: Install dependencies
        run: pip install -r requirements.txt
      
      - name: Run tests
        run: pytest tests/ -v --cov=src --cov-report=xml
      
      - name: Upload coverage
        uses: codecov/codecov-action@v4
```

---

## 3. Secrets & Environment Variables

```yaml
# Access secrets (stored in repo Settings > Secrets)
steps:
  - name: Configure AWS
    env:
      AWS_ACCESS_KEY_ID: ${{ secrets.AWS_ACCESS_KEY_ID }}
      AWS_SECRET_ACCESS_KEY: ${{ secrets.AWS_SECRET_ACCESS_KEY }}
    run: aws s3 ls s3://my-ml-bucket
  
  - name: Call OpenAI API
    env:
      OPENAI_API_KEY: ${{ secrets.OPENAI_API_KEY }}
    run: python evaluate_llm.py
```

**Best practices**:
- Never hardcode secrets in YAML or code
- Use environment-specific secrets (dev, staging, prod)
- Rotate secrets regularly

---

## 4. ML-Specific Workflows

### 4.1 Test ML Code
```yaml
- name: Run ML tests
  run: |
    pytest tests/test_model.py -v
    pytest tests/test_pipeline.py -v
    pytest tests/test_data_validation.py -v
```

### 4.2 Smoke Test Training
```yaml
- name: Smoke test training pipeline
  run: python src/train.py --epochs 1 --sample-size 100 --dry-run
```

### 4.3 Metrics Gate (Block PR if metrics drop)
```yaml
- name: Check model metrics
  run: |
    python scripts/evaluate.py --output metrics.json
    python scripts/assert_metrics.py --min-accuracy 0.85 --min-f1 0.80
```

### 4.4 Report metrics on PR (CML)
```yaml
- name: Publish metrics report
  uses: iterative/setup-cml@v2
- run: |
    echo '## Model Metrics' > report.md
    cat metrics.json >> report.md
    cml comment create report.md
  env:
    REPO_TOKEN: ${{ secrets.GITHUB_TOKEN }}
```

### 4.5 Docker Build & Push
```yaml
- name: Login to GHCR
  uses: docker/login-action@v3
  with:
    registry: ghcr.io
    username: ${{ github.actor }}
    password: ${{ secrets.GITHUB_TOKEN }}

- name: Build and push Docker image
  uses: docker/build-push-action@v5
  with:
    context: .
    push: ${{ github.ref == 'refs/heads/main' }}
    tags: ghcr.io/${{ github.repository }}/ml-api:${{ github.sha }}
    cache-from: type=gha
    cache-to: type=gha,mode=max
```

### 4.6 Deploy on Merge to Main
```yaml
deploy:
  needs: test  # Only deploy if tests pass
  if: github.ref == 'refs/heads/main'
  runs-on: ubuntu-latest
  environment: production
  steps:
    - name: Deploy to AWS
      run: |
        aws ecs update-service --cluster prod --service ml-api \
          --force-new-deployment
```

---

## 5. Reusable Workflows

```yaml
# .github/workflows/reusable-test.yml
on:
  workflow_call:
    inputs:
      python-version:
        type: string
        default: '3.11'

# Call it from another workflow
jobs:
  test:
    uses: ./.github/workflows/reusable-test.yml
    with:
      python-version: '3.11'
```

---

## 6. Self-Hosted Runners (for GPU)

GitHub-hosted runners don't have GPUs. For GPU training:
```yaml
jobs:
  gpu-train:
    runs-on: self-hosted  # Your GPU machine
    container:
      image: pytorch/pytorch:2.2.0-cuda11.8-cudnn8-runtime
```

Setup: install GitHub Actions runner on your GPU server.

---

## 7. Complete ML CI/CD Pipeline

```yaml
# Full ML pipeline: lint → test → train → evaluate → deploy
name: ML CI/CD

on:
  push: {branches: [main]}
  pull_request: {branches: [main]}

jobs:
  lint:
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4
      - uses: actions/setup-python@v5
        with: {python-version: '3.11'}
      - run: pip install ruff black mypy
      - run: ruff check src/
      - run: black --check src/
  
  test:
    needs: lint
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4
      - uses: actions/setup-python@v5
        with: {python-version: '3.11'}
      - run: pip install -r requirements.txt
      - run: pytest tests/ -v
  
  evaluate:
    needs: test
    if: github.event_name == 'pull_request'
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4
      - run: pip install -r requirements.txt
      - run: python scripts/quick_eval.py --output pr_metrics.json
      - run: python scripts/assert_metrics.py
      - uses: actions/upload-artifact@v4
        with:
          name: metrics
          path: pr_metrics.json
  
  deploy:
    needs: test
    if: github.ref == 'refs/heads/main'
    runs-on: ubuntu-latest
    environment: production
    steps:
      - uses: actions/checkout@v4
      - run: echo "Deploying to production..."
```

In [1]:
# ── Generate a GitHub Actions workflow file ─────────────────────
import yaml
from pathlib import Path

def create_ml_workflow(project_name: str, python_version: str = '3.11',
                        min_accuracy: float = 0.80) -> dict:
    """Generate a GitHub Actions workflow for an ML project."""
    return {
        'name': f'{project_name} CI/CD',
        'on': {
            'push': {'branches': ['main']},
            'pull_request': {'branches': ['main']}
        },
        'jobs': {
            'test': {
                'runs-on': 'ubuntu-latest',
                'steps': [
                    {'uses': 'actions/checkout@v4'},
                    {'uses': 'actions/setup-python@v5', 'with': {'python-version': python_version}},
                    {'name': 'Cache pip', 'uses': 'actions/cache@v4', 'with': {
                        'path': '~/.cache/pip',
                        'key': "${{ runner.os }}-pip-${{ hashFiles('requirements.txt') }}"
                    }},
                    {'name': 'Install dependencies', 'run': 'pip install -r requirements.txt'},
                    {'name': 'Lint', 'run': 'ruff check src/ && black --check src/'},
                    {'name': 'Test', 'run': 'pytest tests/ -v --tb=short'},
                    {'name': 'Check metrics', 'run': f'python scripts/check_metrics.py --min-accuracy {min_accuracy}'},
                ]
            }
        }
    }

workflow = create_ml_workflow('MyMLProject', min_accuracy=0.85)
print(yaml.dump(workflow, default_flow_style=False, sort_keys=False))

name: MyMLProject CI/CD
'on':
  push:
    branches:
    - main
  pull_request:
    branches:
    - main
jobs:
  test:
    runs-on: ubuntu-latest
    steps:
    - uses: actions/checkout@v4
    - uses: actions/setup-python@v5
      with:
        python-version: '3.11'
    - name: Cache pip
      uses: actions/cache@v4
      with:
        path: ~/.cache/pip
        key: ${{ runner.os }}-pip-${{ hashFiles('requirements.txt') }}
    - name: Install dependencies
      run: pip install -r requirements.txt
    - name: Lint
      run: ruff check src/ && black --check src/
    - name: Test
      run: pytest tests/ -v --tb=short
    - name: Check metrics
      run: python scripts/check_metrics.py --min-accuracy 0.85



In [2]:
# ── Metrics assertion script (scripts/check_metrics.py) ────────
metrics_script = '''
"""Assert ML metrics meet minimum thresholds for CI gate."""
import argparse, json, sys

def check_metrics(metrics_file: str, min_accuracy: float, min_f1: float):
    with open(metrics_file) as f:
        metrics = json.load(f)
    
    failures = []
    if metrics.get('accuracy', 0) < min_accuracy:
        failures.append(f"accuracy={metrics['accuracy']:.4f} < {min_accuracy}")
    if metrics.get('f1', 0) < min_f1:
        failures.append(f"f1={metrics['f1']:.4f} < {min_f1}")
    
    if failures:
        print("❌ Metrics gate FAILED:")
        for f in failures:
            print(f"  - {f}")
        sys.exit(1)
    else:
        print("✅ All metrics pass thresholds")
        print(f"   accuracy={metrics['accuracy']:.4f}, f1={metrics['f1']:.4f}")

if __name__ == '__main__':
    parser = argparse.ArgumentParser()
    parser.add_argument('--metrics-file', default='metrics.json')
    parser.add_argument('--min-accuracy', type=float, default=0.80)
    parser.add_argument('--min-f1', type=float, default=0.75)
    args = parser.parse_args()
    check_metrics(args.metrics_file, args.min_accuracy, args.min_f1)
'''

# Simulate running
import json
test_metrics = {'accuracy': 0.91, 'f1': 0.88, 'roc_auc': 0.95}
print("Test metrics:", json.dumps(test_metrics, indent=2))
print()
print("Script content:")
print(metrics_script)

Test metrics: {
  "accuracy": 0.91,
  "f1": 0.88,
  "roc_auc": 0.95
}

Script content:

"""Assert ML metrics meet minimum thresholds for CI gate."""
import argparse, json, sys

def check_metrics(metrics_file: str, min_accuracy: float, min_f1: float):
    with open(metrics_file) as f:
        metrics = json.load(f)

    failures = []
    if metrics.get('accuracy', 0) < min_accuracy:
        failures.append(f"accuracy={metrics['accuracy']:.4f} < {min_accuracy}")
    if metrics.get('f1', 0) < min_f1:
        failures.append(f"f1={metrics['f1']:.4f} < {min_f1}")

    if failures:
        print("❌ Metrics gate FAILED:")
        for f in failures:
            print(f"  - {f}")
        sys.exit(1)
    else:
        print("✅ All metrics pass thresholds")
        print(f"   accuracy={metrics['accuracy']:.4f}, f1={metrics['f1']:.4f}")

if __name__ == '__main__':
    parser = argparse.ArgumentParser()
    parser.add_argument('--metrics-file', default='metrics.json')
    parser.add_argument('--min

## Additional Learning Resources

### Official Docs
- [GitHub Actions Documentation](https://docs.github.com/en/actions)
- [GitHub Actions Marketplace](https://github.com/marketplace?type=actions)
- [GitHub Actions for ML GitHub Blog](https://github.blog/2020-06-17-using-github-actions-for-mlops-data-science/)

### MLOps CI/CD
- [CML (Continuous Machine Learning)](https://cml.dev/)
- [DVC CML Tutorial](https://dvc.org/doc/use-cases/ci-cd-for-machine-learning)
- [MLflow CI/CD Guide](https://mlflow.org/docs/latest/deployment/)

### Courses
- [GitHub Actions Course GitHub Skills](https://skills.github.com/)
- [MLOps Zoomcamp DataTalksClub (free)](https://github.com/DataTalksClub/mlops-zoomcamp)

### GitHub
- [Awesome GitHub Actions](https://github.com/sdras/awesome-actions)
- [MLOps Templates](https://github.com/microsoft/MLOps)